# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hidatara-ds/flyrank-intern-ml-gilang/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [2]:
%pip -q install duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    try:
        HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
    except Exception:
        HF_TOKEN = ''

## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [5]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


HTTPException: HTTP Error: HTTP GET error on 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet' (HTTP 0 Internal Server Error)

That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


## Your Turn Implementation

Below, we answer all 3 parts of the **Your Turn** challenge:
1. **90-day window aggregation**: Re-running feature extraction over a 90-day panel window with a `HAVING imp_prev60 >= 100` threshold.
2. **Engineered features**: Adding **position volatility** (`pos_std_prev60` - standard deviation of average position) and **weekend impression share** (`weekend_share_prev60`).
3. **Generalization evaluation**: Comparing **Random Split** (`train_test_split`) vs **Group Split** (`GroupShuffleSplit` on `client_hash_id`) to test out-of-client generalization without leakage.

In [ ]:
# 1. 90-day feature aggregation in DuckDB with Position Volatility and Weekend Impression Share
features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, 
               f.content_hash_id,
               -- Outcome window (last 30 days of the 90-day panel)
               SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               
               -- Feature window (prior 60 days of the 90-day panel: end_d - 90 to end_d - 30)
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev60,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev60,
               AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_prev60,
               
               -- Engineered Feature 1: Position Volatility (Std Dev of daily position in prev 60 days)
               STDDEV_SAMP(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position ELSE NULL END) AS pos_std_prev60,
               
               -- Engineered Feature 2: Weekend Impression Share (Proportion of impressions on weekends in prev 60 days)
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND DAYOFWEEK(f.report_date) IN (0, 6) THEN f.gsc_impressions ELSE 0 END) * 1.0 / 
               NULLIF(SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END), 0) AS weekend_share_prev60
               
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev60 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features_90d):,} content items extracted for 90-day window')
features_90d.head()

In [ ]:
# 2. Merge 90-day features with query-level signals and construct target label
data_90d = features_90d.merge(qsignals, on='content_hash_id', how='left')

# Target label: impression drop > 20% in last 30 days vs average monthly rate in prior 60 days
data_90d['is_declining'] = (data_90d['imp_last30'] < 0.8 * (data_90d['imp_prev60'] / 2.0)).astype(int)

# Handle potential NaNs in engineered features
data_90d['pos_std_prev60'] = data_90d['pos_std_prev60'].fillna(0)
data_90d['weekend_share_prev60'] = data_90d['weekend_share_prev60'].fillna(0)

feature_cols_90d = [
    'imp_prev60', 
    'pos_prev60', 
    'pos_std_prev60', 
    'weekend_share_prev60', 
    'visible_queries', 
    'rare_share', 
    'anon_share', 
    'top_query_share'
]

model_data_90d = data_90d.dropna(subset=feature_cols_90d)
X_90d = model_data_90d[feature_cols_90d]
y_90d = model_data_90d['is_declining']
groups_90d = model_data_90d['client_hash_id']

print(f'Merged modeling data: {len(model_data_90d):,} rows across {groups_90d.nunique()} unique clients')
model_data_90d.head()

In [ ]:
# 3. Model Evaluation: Random Split vs GroupShuffleSplit on client_hash_id
from sklearn.model_selection import GroupShuffleSplit

# A. Random Split (Row-level split with risk of client data leakage)
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X_90d, y_90d, test_size=0.25, random_state=42, stratify=y_90d
)
rf_rand = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_rand, y_tr_rand)

print("=" * 70)
print("1. RANDOM TRAIN/TEST SPLIT (Row-level split - potential client leakage)")
print(f"Base rate (always predict majority): {max(y_te_rand.mean(), 1 - y_te_rand.mean()):.3f}")
print(classification_report(y_te_rand, rf_rand.predict(X_te_rand), digits=3))

# B. Group Split (GroupShuffleSplit by client_hash_id - Honest client holdout)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X_90d, y_90d, groups=groups_90d))

X_tr_grp, X_te_grp = X_90d.iloc[train_idx], X_90d.iloc[test_idx]
y_tr_grp, y_te_grp = y_90d.iloc[train_idx], y_90d.iloc[test_idx]

rf_grp = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_grp, y_tr_grp)

print("=" * 70)
print("2. GROUP SHUFFLE SPLIT (Grouped by client_hash_id - Honest generalization)")
print(f"Base rate (always predict majority): {max(y_te_grp.mean(), 1 - y_te_grp.mean()):.3f}")
print(classification_report(y_te_grp, rf_grp.predict(X_te_grp), digits=3))
print("=" * 70)

### Summary & Insights from Your Turn

1. **Window Extension & Feature Engineering**: Expanding to a **90-day window** allows incorporating multi-month baseline metrics along with richer signals such as **position volatility** (`pos_std_prev60`) and **weekend impression share** (`weekend_share_prev60`).
2. **Generalization Analysis (Random Split vs GroupSplit)**:
   - **Random Split**: Randomly assigning rows across train/test splits lets the model memorize client-specific traits, leading to artificially inflated accuracy metrics due to data leakage across clients.
   - **GroupShuffleSplit**: Grouping by `client_hash_id` ensures that whole clients are held out. This tests the model's capability to generalize to entirely unseen clients, which is essential for real-world deployment.

In [ ]:
# Optional: Local snapshot download (uncomment to download partitions locally)
# from huggingface_hub import snapshot_download
# path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
#                          allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
#                                          'fact_content_daily_performance/month=2026-0*/*.parquet'])